In [ ]:
import torch
import json
import numpy as np
import re
import ast

In [ ]:
from sklearn.metrics import hamming_loss, precision_score, recall_score, f1_score, jaccard_score, accuracy_score

# # 预测结果
# y_pred = [
#     [], [], ['top-left', 'top-center'],
#     ['top-right', 'center-left', 'bottom-left', 'bottom-center', 'bottom-right'],
#     [], [], [], [], [], []
# ]

# # 真实结果
# y_true = [
#     [], [], ['top-left', 'top-center'],
#     ['top-right', 'center-left', 'bottom-left', 'bottom-center', 'bottom-right'],
#     [], [], ['top-center'], ['top-center'], [], []
# ]

def eval_loc(y_pred, y_true):
    # 1. 处理成二进制多标签格式
    labels = ['top-left', 'top-center', 'top-right', 'center-left', 'center-center', 'center-right', 'bottom-left', 'bottom-center', 'bottom-right']
    def to_binary(y, label_list):
        return [[1 if lbl in sample else 0 for lbl in label_list] for sample in y]

    y_true_bin = to_binary(y_true, labels)
    y_pred_bin = to_binary(y_pred, labels)

    # 2. 计算指标
    hamming = hamming_loss(y_true_bin, y_pred_bin)
    precision = precision_score(y_true_bin, y_pred_bin, average='micro')
    recall = recall_score(y_true_bin, y_pred_bin, average='micro')
    f1 = f1_score(y_true_bin, y_pred_bin, average='micro')
    jaccard = jaccard_score(y_true_bin, y_pred_bin, average='samples')
    subset_acc = accuracy_score(y_true_bin, y_pred_bin)

    # 3. 输出结果
    print(f"Hamming Loss: {hamming:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall: {recall:.4f}")
    print(f"F1 Score: {f1:.4f}")
    print(f"Jaccard Similarity: {jaccard:.4f}")
    print(f"Subset Accuracy: {subset_acc:.4f}")

In [ ]:
with open(r'/root/autodl-tmp/Blip2CC/output/BLIP2/Caption_coco_opt2.7b/loc/test_epochbest_rank0.json', 'r') as f:
    data = json.load(f)
with open(r'/root/autodl-tmp/GPT-api/LEVIR-MCI-dataset/LevirCCcaptions-loc.json', 'r') as f:
    data_t = json.load(f)
datajson_t = [x['change_loc'] for x in data_t if x['filepath'] == 'test']

In [ ]:
# 476
results = []
for item in data:
    result = {}
    cap = item['caption']
    result['image_id'] = item['image_id']
    result['change_loc'] = {}
    if 'roads [] buildings []' in cap:       
        result['change_loc']['road'] = []
        result['change_loc']['building'] = []
        results.append(result)
        continue
    # print(item['image_id'])
    # if item['image_id']==476 or item['image_id']==484 or item['image_id']==705 or item['image_id']==729 or item['image_id']==1093 or item['image_id']==1219 or item['image_id']==1231 or item['image_id']==1245 or item['image_id']==1264 or item['image_id']==1265 or item['image_id']==1619:
    #     continue
    r_matches = re.findall(r'roads\s*(\[[^\]]*\])', cap)
    # print(r_matches)
    result['change_loc']['road'] = r_matches[0]
    if r_matches[0] !=[]:
        result['change_loc']['road'] = ast.literal_eval(r_matches[0])
    b_matches = re.findall(r'buildings\s*(\[[^\]]*\])', cap)
    result['change_loc']['building'] = b_matches[0]
    if b_matches[0] !=[]:
        result['change_loc']['building'] = ast.literal_eval(b_matches[0])
    results.append(result)

In [ ]:
## 道路
r_truth = [x['roads'] for x in datajson_t]
r_pred = [x['change_loc']['road'] for x in results]
print('='*10, 'road', '='*10)
eval_loc(r_pred, r_truth)

## 建筑
b_truth = [x['buildings'] for x in datajson_t]
b_pred = [x['change_loc']['building'] for x in results]
print('='*10, 'building', '='*10)
eval_loc(b_pred, b_truth)
